In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ tanh(z) = t = \frac{e^z - e^{-z}}{e^z + e^{-z}} = $$
$$ \frac{e^z}{e^z} \cdot \frac{e^z - e^{-z}}{e^z + e^{-z}} = $$
$$ \frac{2e^{2z}}{e^{2z} + 1} - \frac{e^{2z} + 1}{e^{2z} + 1} = $$
$$ 2 \cdot \text{sigmoid}(2z) - 1 $$
$$ \frac{\partial t}{\partial z} = 1 - t^2 $$

$$ \diamond \diamond \diamond $$

$$ t \in (-1, 1), p \in (0, 1) $$
$$ p = 0.5t + 0.5 $$
$$ t = 2p - 1 $$
$$ \frac{\partial p}{\partial z} = 0.5(1 - (2p-1)^2) $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import sigmoid # type: ignore
from approx import approx # type: ignore

def tanh(x: tr.Tensor) -> tr.Tensor:
    """Hyperbolic tangent function."""

    return 2 * sigmoid.sigmoid(2 * x) - 1
    

class TanhFunction(autograd.Function):
    """Custom autograd function for the hyperbolic tangent function."""

    @staticmethod
    def forward(ctx, x: tr.Tensor) -> tr.Tensor:
        t = tanh(x)  
        assert t.shape == x.shape

        ctx.save_for_backward(t)
        return t

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, ]:
        (t, ) = ctx.saved_tensors

        df_dx = 1 - t ** 2
        assert df_dx.shape == t.shape
        
        dF_dx = dF_df * df_dx
        assert dF_dx.shape == t.shape

        return (dF_dx, )
    

class Tanh(nn.Module):
    """Custom module for the hyperbolic tangent function."""

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return TanhFunction.apply(x)


def test_tanh_1() -> None:
    assert tanh(0) == approx(tr.tensor(0.0))
    assert tanh(1000) == approx(tr.tensor(1.0))
    assert tanh(-1000) == approx(tr.tensor(-1.0))
    assert tanh(0) == approx(tr.tensor(0.0))
    assert tanh(1) == approx(tr.tensor(0.76159))
    assert tanh(-1) == approx(tr.tensor(-0.76159))


def test_tanh_2() -> None:
    tr.manual_seed(0)

    samples = 10
    x = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = TanhFunction.apply(x1)
    F1 = y1.sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = tr.tanh(x2)
    F2 = y2.sum()
    F2.backward()

    assert y1 == approx(y2)
    assert x1.grad == approx(x2.grad)


if __name__ == "__main__":
    test_tanh_1()
    test_tanh_2()
